# Hyperparameter Tuning Task

## Objective
In this lab, you will:
- train a baseline model
- tune key hyperparameters
- compare results with the baseline
- identify the best model


## Import Libraries

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

## Load and Prepare the Dataset

In [2]:
# Load Fashion MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# TODO:
# 1) Normalize x_train and x_test
# 2) Flatten the images from (28, 28) to (784,)
x_train=x_train.reshape(-1,784).astype('float32')/255.0
x_test=x_test.reshape(-1,784).astype(np.float32)/255.0

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


## Task 1: Build a Baseline Model

In [3]:
def build_baseline_model():
    # TODO:
    # Build a simple neural network with:
    # - one hidden Dense layer with 128 units and ReLU
    # - one output Dense layer with 10 units and softmax
    model=keras.Sequential([
        keras.layers.Dense(256, activation="relu", input_shape=(784,)),
        keras.layers.Dense(128,activation='relu'),
        keras.layers.Dense(10,activation='softmax')
    ])
    # Compile the model using:
    # - optimizer = 'adam'
    # - loss = 'sparse_categorical_crossentropy'
    # - metrics = ['accuracy']
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [4]:
# Train the baseline model
baseline_model = build_baseline_model()

history_baseline = baseline_model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

baseline_loss, baseline_acc = baseline_model.evaluate(x_test, y_test)
print("Baseline Accuracy:", baseline_acc)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.8220 - loss: 0.4961 - val_accuracy: 0.8602 - val_loss: 0.3921
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.8615 - loss: 0.3722 - val_accuracy: 0.8490 - val_loss: 0.3956
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.8767 - loss: 0.3326 - val_accuracy: 0.8815 - val_loss: 0.3306
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.8854 - loss: 0.3080 - val_accuracy: 0.8759 - val_loss: 0.3457
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.8929 - loss: 0.2881 - val_accuracy: 0.8765 - val_loss: 0.3436
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8707 - loss: 0.3628
Baseline Accuracy: 0.8707000017166138


## Task 2: Tune Learning Rate

In [5]:
def build_model_lr(lr):
    # TODO:
    # Build the same model as baseline
    # but use Adam with a custom learning rate

    model = keras.Sequential([
        keras.layers.Dense(128,activation='relu'),
        keras.layers.Dense(10,activation='softmax')
        ])

    optimizer = keras.optimizers.Adam(learning_rate=lr)

    # Compile the model here
    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [6]:
learning_rates = [0.1, 0.01, 0.001]

for lr in learning_rates:
    print(f"\nTraining with learning rate = {lr}")

    model = build_model_lr(lr)
    model.fit(x_train, y_train, epochs=3, batch_size=32, verbose=0)
    loss, acc = model.evaluate(x_test, y_test, verbose=0)

    print("Accuracy:", acc)


Training with learning rate = 0.1
Accuracy: 0.10000000149011612

Training with learning rate = 0.01
Accuracy: 0.8395000100135803

Training with learning rate = 0.001
Accuracy: 0.8719000220298767


## Task 3: Tune Number of Layers

In [7]:
def build_model_layers(num_layers):
    model = keras.Sequential()
    if num_layers >= 1:
        model.add(keras.layers.Dense(256, activation='relu', input_shape=(784,)))
        for _ in range(num_layers - 1):
            model.add(keras.layers.Dense(128, activation='relu'))
    else:
        model.add(keras.layers.Input(shape=(784,)))

    model.add(keras.layers.Dense(10, activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [8]:
for layers_num in [1, 2, 3]:
    print(f"\nTraining with {layers_num} hidden layer(s)")
    model = build_model_layers(layers_num)
    model.fit(x_train, y_train, epochs=3, batch_size=32, verbose=0)
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    print("Accuracy:", acc)


Training with 1 hidden layer(s)
Accuracy: 0.8659999966621399

Training with 2 hidden layer(s)
Accuracy: 0.875

Training with 3 hidden layer(s)
Accuracy: 0.8671000003814697


## Task 4: Tune Batch Size

In [9]:
batch_sizes = [16, 32, 64]

for batch in batch_sizes:
    print(f"\nTraining with batch size = {batch}")
    model = build_baseline_model()
    model.fit(x_train, y_train, epochs=3, batch_size=batch, verbose=0)
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    print("Accuracy:", acc)


Training with batch size = 16
Accuracy: 0.8669999837875366

Training with batch size = 32
Accuracy: 0.8759999871253967

Training with batch size = 64
Accuracy: 0.8697999715805054


## Task 5: Add Momentum

In [10]:
def build_model_momentum():
    model = keras.Sequential([
        keras.Input(shape=(784,)),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ])

    optimizer = keras.optimizers.SGD(learning_rate=0.01,momentum=0.9)

    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

model = build_model_momentum()
model.fit(x_train, y_train, epochs=5, batch_size=32)
loss, acc = model.evaluate(x_test, y_test)
print("Accuracy with Momentum:", acc)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8146 - loss: 0.5235
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8575 - loss: 0.3976
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8688 - loss: 0.3592
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8762 - loss: 0.3349
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8826 - loss: 0.3183
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8621 - loss: 0.3754
Accuracy with Momentum: 0.8621000051498413


## Final Comparison

In [12]:
# TODO:
# 1) Record the best result you achieved
# 2) Compare it with the baseline accuracy
# 3) Write a short conclusion

best_model_acc = 0.8760   # Replace with your best accuracy

print("Baseline Accuracy:", baseline_acc)
print("Best Model Accuracy:", best_model_acc)

Baseline Accuracy: 0.8707000017166138
Best Model Accuracy: 0.876


In [ ]:
# the best accuracy I achieved was 87.60% using a batch size of 32, compared to the baseline accuracy of 87.07%
# this shows a small improvement in performance. I also noticed a high learning rate of 0.1 that significantly reduced accuracy,
# while adding momentum did not improve the results in this experiment

## Reflection Questions
1. Which hyperparameter had the biggest impact on accuracy?
2. Did adding more layers always improve performance?
3. How did batch size affect training speed and results?
4. What is the advantage of using momentum?
5. Which model would you choose, and why?


1. The learning rate had the biggest impact on accuracy. A learning rate of 0.1 dropped the accuracy to 10%, while 0.001 achieved 87.19%.
2. No, adding more layers did not always improve performance. Two hidden layers achieved the highest accuracy of 87.50%, while three layers performed slightly worse.
3. Batch size affected the number of updates during training. Larger batch sizes generally require fewer updates per epoch. A batch size of 32 gave the best accuracy at 87.60%.
4. Momentum helps speed up training and reduce fluctuations by using previous updates. However, it did not improve accuracy in my experiment.
5. I would choose the model with a batch size of 32 because it achieved the highest test accuracy of 87.60% among my experiments.